# Week 06B — Deep Learning II: Training Well and Transfer Learning

**MCTA 4364 Machine Vision** · Session 2 of Week 6 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W06_deep_learning/W06B_training_transfer.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W06_deep_learning/W06B_training_transfer.ipynb)

In Session A you trained a CNN on 60,000 tidy 28×28 images. Real machine-vision projects look different: **colour photos,
a few hundred labelled images, and a deadline.** This session covers the practical skills that make deep learning work
in that setting, and ends by revisiting the Week 4–5 failure cases with learned features.

### Learning outcomes
By the end of this session you will be able to:
1. Organise data into **train / validation / test** splits and explain the role of each.
2. Diagnose **overfitting** from training curves and reduce it with **augmentation**, **dropout** and **weight decay**.
3. Choose an **optimiser** and **learning-rate schedule**, and debug training with the *overfit-one-batch* test.
4. Apply **transfer learning** (linear probe vs fine-tuning) and quantify how much labelled data it saves.
5. Explain a CNN's decision with **Grad-CAM**, implemented from scratch.
6. Compare hand-crafted and learned features on **retrieval** and **segmentation**, and check that a model fits a **4 GB GPU**.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–5 | 1 | Recap and the real-data problem |
| 5–20 | 2–3 | Data splits, normalisation, augmentation playground |
| 20–35 | 4–5 | Overfitting lab; optimisers, schedules, debugging |
| 35–55 | 6 | Transfer learning: scratch vs linear probe vs fine-tune; data efficiency |
| 55–67 | 7 | Grad-CAM: what is the network looking at? |
| 67–75 | 8–9 | ViT in one page; classical vs learned revisited |
| 75–80 | 13 | Quiz and exit ticket |
| Home | 10–12 | 4 GB GPU check, exercises with self-checks, webcam challenge |

**Prerequisites:** Week 06A.

**Hardware:** about 10 minutes on a 4 GB NVIDIA laptop GPU; about 15 minutes on a CPU with reduced settings.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import tarfile
import time
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import skimage.data
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image
from skimage.feature import hog
from skimage.filters import threshold_otsu
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2 as T
from torchvision.transforms.v2 import functional as TF

from cvhelpers import SMOKE, check, data_dir, download, interact, pipeline_diagram, quiz, youtube
from dlhelpers import (IMAGENET_MEAN, IMAGENET_STD, confusion_matrix, count_params, denormalize, fit,
                       hw_profile, load_pretrained, plot_confusion, plot_history, predict, set_seed)

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
HW = hw_profile()
DEVICE = HW["device"]
set_seed(0)

# Sizes adapt to your hardware (CI uses the tiny SMOKE setting).
if SMOKE:
    PER_CLASS_TRAIN, PER_CLASS_VAL, IMG, EPOCHS_SMALL, EPOCHS_TL = 20, 10, 96, 2, 1
elif HW["cuda"]:
    PER_CLASS_TRAIN, PER_CLASS_VAL, IMG, EPOCHS_SMALL, EPOCHS_TL = 200, 50, 128, 30, 6
else:
    PER_CLASS_TRAIN, PER_CLASS_VAL, IMG, EPOCHS_SMALL, EPOCHS_TL = 60, 30, 112, 15, 3
BATCH = 16 if SMOKE else int(64 * HW["batch_scale"])
print(f"{PER_CLASS_TRAIN} train / {PER_CLASS_VAL} val / {PER_CLASS_VAL} test images per class | "
      f"image size {IMG} | batch {BATCH} | epochs {EPOCHS_SMALL} (small CNN), {EPOCHS_TL} (transfer)")

## 1. Recap and the real-data problem

Session A's recipe was **model + loss + gradient descent**. With plenty of clean data that is enough. With a few hundred
real photos, a network with millions of parameters will simply **memorise** the training set. Today's tools all fight that problem:

In [ ]:
pipeline_diagram(["Collect &\nsplit data", "Augment", "Pretrained\nbackbone", "Train with\nregularisation", "Validate &\nearly-stop",
                  "Explain\n(Grad-CAM)", "Test once"],
                 colors=["#e8f0fe", "#fce8b2", "#ceead6", "#fce8b2", "#e8f0fe", "#fad2cf", "#f1f3f4"],
                 notes=["Sec. 2", "Sec. 3", "Sec. 6", "Sec. 4-5", "Sec. 4", "Sec. 7", "Sec. 6"],
                 title="Figure 1.1 - A practical deep-learning workflow for machine vision", figsize=(17, 2.4))

## 2. Real photos and honest evaluation

### 2.1 The dataset
We use **Imagenette** (fast.ai, Apache-2.0 licence): a 10-class subset of ImageNet with real, varied colour photos.
The 160-pixel version is a 99 MB download, cached in `data/`. We deliberately use only a **small subset** per class
to mimic a typical industrial project, where labelling is expensive.

In [ ]:
SYNSETS = {"n01440764": "tench", "n02102040": "springer spaniel", "n02979186": "cassette player",
           "n03000684": "chain saw", "n03028079": "church", "n03394916": "French horn",
           "n03417042": "garbage truck", "n03425413": "gas pump", "n03445777": "golf ball", "n03888257": "parachute"}
CLASSES = list(SYNSETS.values())

root = data_dir()
folder = root / "imagenette2-160"
if not folder.exists():
    tgz = download("https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz", root / "imagenette2-160.tgz")
    with tarfile.open(tgz) as tar:
        try:
            tar.extractall(root, filter="data")
        except TypeError:  # Python < 3.11.4
            tar.extractall(root)


def load_split(split, per_class, offset=0, seed=0):
    """Load `per_class` images per class (after skipping `offset`) into memory."""
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for c, syn in enumerate(SYNSETS):
        files = sorted((folder / split / syn).glob("*.JPEG"))
        chosen = rng.permutation(len(files))[offset:offset + per_class]
        for i in chosen:
            images.append(Image.open(files[i]).convert("RGB"))
            labels.append(c)
    return images, np.array(labels)


train_imgs, train_labels = load_split("train", PER_CLASS_TRAIN)
val_imgs, val_labels = load_split("val", PER_CLASS_VAL)                            # for model selection
test_imgs, test_labels = load_split("val", PER_CLASS_VAL, offset=PER_CLASS_VAL)    # touched once, at the end
full_counts = {s: [len(list((folder / s / syn).glob("*.JPEG"))) for syn in SYNSETS] for s in ("train", "val")}
print(f"train {len(train_imgs)} | val {len(val_imgs)} | test {len(test_imgs)} images")

In [ ]:
fig, axes = plt.subplots(10, 7, figsize=(12, 17))
for c in range(10):
    idx = np.where(train_labels == c)[0][:7]
    for j, i in enumerate(idx):
        axes[c, j].imshow(train_imgs[i]); axes[c, j].set_xticks([]); axes[c, j].set_yticks([])
    axes[c, 0].set_ylabel(CLASSES[c], rotation=0, ha="right", va="center", fontsize=10)
plt.suptitle("Figure 2.1 - Imagenette: real photos with clutter, scale and viewpoint changes", fontweight="bold")
plt.tight_layout(); plt.show()

### 2.2 Train / validation / test

| Split | Used for | Rule |
|---|---|---|
| **Train** | computing gradients | the model sees it every epoch |
| **Validation** | choosing hyper-parameters, early stopping, comparing models | never train on it |
| **Test** | the final, unbiased performance estimate | use it **once**, after all decisions are made |

If you tune anything on the test set, it silently becomes a validation set and your reported accuracy becomes optimistic.
In industry, the test set should also come from the **deployment conditions**: the real camera, lighting and production line.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 3.6), gridspec_kw={"width_ratios": [1.3, 1]})
n = [len(train_imgs), len(val_imgs), len(test_imgs)]
left = 0
for size, name, color in zip(n, ["train", "validation", "test"], ["#1a73e8", "#fbbc04", "#d93025"]):
    axes[0].barh(0, size, left=left, color=color, edgecolor="white")
    axes[0].text(left + size / 2, 0, f"{name}\n{size} images", ha="center", va="center", color="white", fontweight="bold")
    left += size
axes[0].set_xlim(0, left); axes[0].axis("off"); axes[0].set_title("Our splits (a small, realistic budget)")
xs = np.arange(10)
axes[1].bar(xs - 0.2, full_counts["train"], 0.4, label="full train folder", color="#8ab4f8")
axes[1].bar(xs + 0.2, full_counts["val"], 0.4, label="full val folder", color="#fdd663")
axes[1].set_xticks(xs, CLASSES, rotation=60, ha="right", fontsize=8); axes[1].legend(fontsize=8)
axes[1].set_title("Images available per class (roughly balanced)")
plt.suptitle("Figure 2.2 - Data splits and class balance", fontweight="bold")
plt.tight_layout(); plt.show()

### 2.3 Normalisation
Networks train best when inputs are centred and scaled. Pretrained ImageNet models expect each colour channel to be
normalised with the ImageNet statistics, $x' = (x - \mu_c)/\sigma_c$ with $\mu = (0.485, 0.456, 0.406)$ and $\sigma = (0.229, 0.224, 0.225)$.
Using different statistics from the ones a model was trained with is a classic silent bug.

## 3. Data augmentation

Augmentation creates new, *plausible* training images by transforming the ones you have. The label must stay correct
after the transform. That is a design decision that depends on your application:

| Transform | Good for | Dangerous when |
|---|---|---|
| horizontal flip | most natural objects | text, left/right parts, asymmetric components |
| rotation | top-down views (PCBs, cells) | orientation *is* the defect, gauges and dials |
| crop / scale | objects at varying distance | a defect near the border gets cropped out |
| colour jitter | changing illumination | colour is the class (wire colours, ripeness) |
| blur / noise | cheap cameras, motion | fine scratches are the defect |

In [ ]:
normalize = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
to_float = [T.ToImage(), T.ToDtype(torch.float32, scale=True), normalize]
train_tf = T.Compose([T.RandomResizedCrop(IMG, scale=(0.5, 1.0), antialias=True), T.RandomHorizontalFlip(),
                      T.ColorJitter(0.3, 0.3, 0.3, 0.05), *to_float])
eval_tf = T.Compose([T.Resize(IMG, antialias=True), T.CenterCrop(IMG), *to_float])

sample = train_imgs[np.where(train_labels == 6)[0][0]]
fig, axes = plt.subplots(2, 7, figsize=(17, 5.4))
axes[0, 0].imshow(eval_tf(sample).permute(1, 2, 0).mul(torch.tensor(IMAGENET_STD)).add(torch.tensor(IMAGENET_MEAN)).clamp(0, 1))
axes[0, 0].set_title("original", fontweight="bold")
for ax in axes.ravel()[1:]:
    ax.imshow(denormalize(train_tf(sample)))
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Figure 3.1 - 13 random augmentations of one training image: the network never sees the same picture twice",
             fontweight="bold")
plt.tight_layout(); plt.show()

batch = torch.stack([eval_tf(im) for im in train_imgs[:64]])
raw = batch * torch.tensor(IMAGENET_STD)[:, None, None] + torch.tensor(IMAGENET_MEAN)[:, None, None]
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))
for c, color in enumerate(["#d93025", "#188038", "#1a73e8"]):
    axes[0].hist(raw[:, c].ravel(), bins=60, color=color, alpha=0.5, label="RGB"[c])
    axes[1].hist(batch[:, c].ravel(), bins=60, color=color, alpha=0.5, label="RGB"[c])
axes[0].set_title("pixel values before normalisation [0, 1]"); axes[1].set_title("after ImageNet normalisation (about zero mean, unit std)")
for ax in axes:
    ax.legend(); ax.set_yticks([])
plt.suptitle("Figure 3.2 - Per-channel normalisation", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: augmentation playground
Choose an image and combine transforms. Ask yourself: *would the label still be correct for my inspection task?*

In [ ]:
def aug_demo(index=0, rotation=0, brightness=1.0, contrast=1.0, saturation=1.0, hue=0.0, crop_scale=1.0,
             hflip=False, vflip=False, blur=0):
    img = train_imgs[index]
    out = TF.rotate(img, rotation)
    w, h = out.size
    cw, ch = int(w * np.sqrt(crop_scale)), int(h * np.sqrt(crop_scale))
    out = TF.center_crop(out, [ch, cw])
    out = TF.adjust_brightness(out, brightness); out = TF.adjust_contrast(out, contrast)
    out = TF.adjust_saturation(out, saturation); out = TF.adjust_hue(out, hue)
    if hflip: out = TF.horizontal_flip(out)
    if vflip: out = TF.vertical_flip(out)
    if blur: out = TF.gaussian_blur(out, 2 * blur + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
    axes[0].imshow(img); axes[0].set_title(f"original: {CLASSES[train_labels[index]]}")
    axes[1].imshow(out); axes[1].set_title("augmented")
    for ax in axes:
        ax.axis("off")
    plt.show()


interact(aug_demo, index=widgets.IntSlider(0, 0, len(train_imgs) - 1, continuous_update=False),
         rotation=widgets.IntSlider(0, -180, 180, 5), brightness=widgets.FloatSlider(1.0, min=0.2, max=2.0, step=0.05),
         contrast=widgets.FloatSlider(1.0, min=0.2, max=2.0, step=0.05),
         saturation=widgets.FloatSlider(1.0, min=0.0, max=2.0, step=0.05),
         hue=widgets.FloatSlider(0.0, min=-0.5, max=0.5, step=0.02), crop_scale=widgets.FloatSlider(1.0, min=0.2, max=1.0, step=0.05),
         hflip=False, vflip=False, blur=widgets.IntSlider(0, 0, 10))

## 4. Overfitting and regularisation

**Overfitting** means training accuracy keeps rising while validation accuracy stalls or falls: the network memorises
individual training images instead of learning general rules. The gap between the two curves is the **generalisation gap**.

We train the same small CNN (at 64×64 pixels, so it is fast) twice:
- **baseline**: no augmentation, no regularisation;
- **regularised**: augmentation + **dropout** (randomly zeroing 30% of the classifier inputs during training, so no single
  feature can be relied upon) + **weight decay** (a penalty $\lambda\lVert\mathbf{w}\rVert^2$ that keeps weights small, via `AdamW`).

The network also uses **batch normalisation** (Ioffe & Szegedy, 2015) after each convolution, which re-centres activations
and makes training faster and more stable (Figure 4.3).

In [ ]:
class ImageList(Dataset):
    def __init__(self, images, labels, transform):
        self.images, self.labels, self.transform = images, labels, transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        return self.transform(self.images[i]), int(self.labels[i])


def make_loaders(tf_train, tf_eval, batch=BATCH):
    return (DataLoader(ImageList(train_imgs, train_labels, tf_train), batch_size=batch, shuffle=True,
                       num_workers=HW["num_workers"], pin_memory=HW["cuda"]),
            DataLoader(ImageList(val_imgs, val_labels, tf_eval), batch_size=batch * 2,
                       num_workers=HW["num_workers"]))


def small_cnn(dropout=0.0):
    def block(c_in, c_out):
        return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2))
    return nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(256, len(CLASSES)))


SMALL = 64
plain_tf = T.Compose([T.Resize(SMALL, antialias=True), T.CenterCrop(SMALL), *to_float])
aug_tf = T.Compose([T.RandomResizedCrop(SMALL, scale=(0.5, 1.0), antialias=True), T.RandomHorizontalFlip(),
                    T.ColorJitter(0.3, 0.3, 0.3, 0.05), *to_float])

set_seed(0)
m_plain = small_cnn(0.0).to(DEVICE)
print("Baseline (no augmentation, no regularisation):")
h_plain = fit(m_plain, *make_loaders(plain_tf, plain_tf), torch.optim.Adam(m_plain.parameters(), 1e-3),
              EPOCHS_SMALL, DEVICE, amp=HW["amp"])
set_seed(0)
m_reg = small_cnn(0.3).to(DEVICE)
print("Regularised (augmentation + dropout 0.3 + weight decay):")
h_reg = fit(m_reg, *make_loaders(aug_tf, plain_tf), torch.optim.AdamW(m_reg.parameters(), 1e-3, weight_decay=0.05),
            EPOCHS_SMALL, DEVICE, amp=HW["amp"])

In [ ]:
plot_history({"baseline": h_plain, "regularised": h_reg},
             title="Figure 4.1 - Overfitting: dashed = train, solid = validation")
fig, ax = plt.subplots(figsize=(8, 3.6))
gaps = [h_plain["train_acc"][-1] - h_plain["val_acc"][-1], h_reg["train_acc"][-1] - h_reg["val_acc"][-1]]
vals = [h_plain["val_acc"][-1], h_reg["val_acc"][-1]]
x = np.arange(2)
ax.bar(x - 0.2, gaps, 0.4, color="#d93025", label="generalisation gap (train - val acc)")
ax.bar(x + 0.2, vals, 0.4, color="#188038", label="validation accuracy")
ax.set_xticks(x, ["baseline", "regularised"]); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("Figure 4.2 - Regularisation shrinks the gap", fontweight="bold")
plt.show()

In [ ]:
# Figure 4.3: why batch normalisation helps - activations in a 10-layer random network
torch.manual_seed(0)
x0 = torch.randn(512, 256)
fig, axes = plt.subplots(2, 5, figsize=(17, 5), sharey="row")
for row, use_bn in enumerate([False, True]):
    h = x0
    for layer in range(10):
        h = nn.Linear(256, 256)(h)
        if use_bn:
            h = nn.BatchNorm1d(256)(h)
        h = torch.tanh(h)
        if layer % 2 == 1:
            ax = axes[row, layer // 2]
            ax.hist(h.detach().ravel(), bins=50, color="#1a73e8" if use_bn else "#d93025")
            ax.set_title(f"layer {layer + 1}: std {h.std():.3f}", fontsize=9)
    axes[row, 0].set_ylabel("with BatchNorm" if use_bn else "without")
plt.suptitle("Figure 4.3 - Without normalisation activations shrink layer by layer (gradients vanish); BatchNorm keeps them healthy",
             fontweight="bold")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
rng = np.random.default_rng(3)
for ax, p, title in [(axes[0], 0.0, "standard network"), (axes[1], 0.4, "with dropout (p = 0.4), one training step")]:
    layers = [4, 6, 6, 2]
    pos = {(l, i): (l * 2.0, i - n / 2) for l, n in enumerate(layers) for i in range(n)}
    dropped = {k for k in pos if 0 < k[0] < 3 and rng.random() < p}
    for (l, i), (x1, y1) in pos.items():
        if l + 1 < len(layers):
            for j in range(layers[l + 1]):
                x2, y2 = pos[(l + 1, j)]
                if (l, i) not in dropped and (l + 1, j) not in dropped:
                    ax.plot([x1, x2], [y1, y2], color="#9aa0a6", lw=0.6)
        ax.add_patch(plt.Circle((x1, y1), 0.28, fc="#f1f3f4" if (l, i) in dropped else "#8ab4f8", ec="k"))
        if (l, i) in dropped:
            ax.text(x1, y1, "x", ha="center", va="center", color="#d93025", fontsize=14, fontweight="bold")
    ax.set_xlim(-0.6, 6.6); ax.set_ylim(-3.6, 3.2); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title)
plt.suptitle("Figure 4.4 - Dropout randomly removes units during training (all units are used at test time)", fontweight="bold")
plt.tight_layout(); plt.show()

## 5. Optimisers, learning-rate schedules and debugging

| Optimiser | Update idea | Typical learning rate |
|---|---|---|
| SGD | $\mathbf{w} \leftarrow \mathbf{w} - \eta\nabla\mathcal{L}$ | 0.01–0.1 |
| SGD + momentum | follows a running average of gradients | 0.01–0.1, momentum 0.9 |
| Adam / AdamW | per-weight adaptive steps (+ decoupled weight decay) | 1e-4 – 3e-3 |

A **learning-rate schedule** starts with larger steps and ends with small ones to settle into a good minimum.
**Warm-up** (a few epochs of increasing $\eta$) protects pretrained weights and large batches from an early shock.

In [ ]:
set_seed(0)
opt_runs = {}
n_ep = max(2, EPOCHS_SMALL // 2)
for name, make_opt in {"SGD": lambda p: torch.optim.SGD(p, lr=0.01),
                       "SGD + momentum": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
                       "AdamW": lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=0.05)}.items():
    set_seed(0)
    m = small_cnn(0.3).to(DEVICE)
    opt_runs[name] = fit(m, *make_loaders(aug_tf, plain_tf), make_opt(m.parameters()), n_ep, DEVICE,
                         amp=HW["amp"], verbose=False)
    print(f"{name:15s} final val acc {opt_runs[name]['val_acc'][-1]:.3f}")
plot_history(opt_runs, title="Figure 5.1 - Same network, same data, three optimisers")

In [ ]:
def schedule(make, epochs=50):
    p = torch.nn.Parameter(torch.zeros(1))
    opt = torch.optim.SGD([p], lr=0.1)
    sched = make(opt)
    lrs = []
    for _ in range(epochs):
        lrs.append(opt.param_groups[0]["lr"]); opt.step(); sched.step()
    return lrs


S = torch.optim.lr_scheduler
schedules = {
    "constant": lambda o: S.LambdaLR(o, lambda e: 1.0),
    "step (x0.1 every 20)": lambda o: S.StepLR(o, 20, 0.1),
    "cosine": lambda o: S.CosineAnnealingLR(o, 50),
    "warm-up 5 + cosine": lambda o: S.SequentialLR(o, [S.LinearLR(o, 0.05, 1.0, 5), S.CosineAnnealingLR(o, 45)], [5]),
    "one-cycle": lambda o: S.OneCycleLR(o, max_lr=0.1, total_steps=50),
}
fig, ax = plt.subplots(figsize=(10, 4))
for name, make in schedules.items():
    ax.plot(schedule(make), lw=2.2, label=name)
ax.set_xlabel("epoch"); ax.set_ylabel("learning rate"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Figure 5.2 - Common learning-rate schedules", fontweight="bold")
plt.show()

### Debugging tip: overfit one batch first
Before a long training run, check that the model can drive the loss to almost zero on a **single small batch**
(Karpathy, 2019). If it cannot, there is a bug in the model, the labels or the loss, and no amount of training will fix it.

In [ ]:
set_seed(0)
m = small_cnn(0.0).to(DEVICE)
xb, yb = next(iter(make_loaders(plain_tf, plain_tf, batch=16)[0]))
xb, yb = xb.to(DEVICE), yb.to(DEVICE)
opt = torch.optim.Adam(m.parameters(), 1e-3)
losses = []
for step in range(30 if SMOKE else 100):
    loss = F.cross_entropy(m(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
plt.figure(figsize=(8, 3.2)); plt.semilogy(losses, lw=2); plt.grid(alpha=0.3)
plt.xlabel("step"); plt.ylabel("loss (log)")
plt.title(f"Figure 5.3 - Overfitting one batch of 16 images: loss {losses[0]:.2f} -> {losses[-1]:.4f}", fontweight="bold")
plt.show()

## 6. Transfer learning

A network trained on **ImageNet** (1.28 million images, 1,000 classes) has learned a general visual vocabulary. Early layers
hold edge and colour detectors, middle layers hold textures and parts, and late layers hold object-level concepts (Zeiler & Fergus, 2014;
Yosinski et al., 2014). We can **reuse** that vocabulary for our own task:

| Strategy | What is trained | When to use |
|---|---|---|
| **From scratch** | everything, from random weights | huge dataset, or a very unusual domain |
| **Linear probe** (feature extraction) | only a new final layer; the backbone is frozen | very little data; fast; runs on a CPU |
| **Fine-tuning** | everything, starting from pretrained weights, with a small learning rate | moderate data; usually the best accuracy |

We use **ResNet-18** (He et al., 2016): 11.7M parameters and 1.8 GFLOPs per 224×224 image, which fine-tunes comfortably on a 4 GB GPU.
The weights (45 MB) download on first use.

In [ ]:
pipeline_diagram(["ImageNet\n1.28M images", "ResNet-18\npretrained", "remove 1000-class\nhead", "frozen or\nfine-tuned backbone",
                  "new 10-class\nhead", "your task"],
                 colors=["#f1f3f4", "#ceead6", "#fad2cf", "#ceead6", "#fce8b2", "#e8f0fe"],
                 title="Figure 6.1 - Transfer learning: reuse the backbone, replace the head", figsize=(16, 2.3))

WEIGHTS = torchvision.models.ResNet18_Weights.DEFAULT
imnet_model, PRETRAINED = load_pretrained(torchvision.models.resnet18, WEIGHTS)
imnet_model = imnet_model.to(DEVICE).eval()
IMNET_CLASSES = WEIGHTS.meta["categories"]

w = imnet_model.conv1.weight.detach().cpu()
w = (w - w.amin((1, 2, 3), keepdim=True)) / (w.amax((1, 2, 3), keepdim=True) - w.amin((1, 2, 3), keepdim=True))
fig, axes = plt.subplots(4, 16, figsize=(16, 4.4))
for ax, k in zip(axes.ravel(), w):
    ax.imshow(k.permute(1, 2, 0)); ax.axis("off")
plt.suptitle("Figure 6.2 - The 64 first-layer filters (7x7, RGB) of ImageNet-pretrained ResNet-18: oriented edges, "
             "colour blobs, Gabor-like stripes" + ("" if PRETRAINED else "  [random weights: offline]"), fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
def make_resnet(strategy):
    if strategy == "scratch":
        model = torchvision.models.resnet18(weights=None)
    else:
        model, _ = load_pretrained(torchvision.models.resnet18, WEIGHTS)
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))   # new, randomly initialised head
    if strategy == "linear probe":
        for name, p in model.named_parameters():
            p.requires_grad = name.startswith("fc.")
        # Note: BatchNorm layers still update their running statistics in train mode;
        # a stricter probe would also keep them in eval mode.
    return model.to(DEVICE)


strategies = {"scratch": 1e-3, "linear probe": 1e-3, "fine-tune": 3e-4}
tl_hist, tl_time, tl_trainable, tl_models = {}, {}, {}, {}
for name, lr in strategies.items():
    set_seed(0)
    model = make_resnet(name)
    params = [p for p in model.parameters() if p.requires_grad]
    tl_trainable[name] = sum(p.numel() for p in params)
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS_TL)
    print(f"--- {name}: {tl_trainable[name]:,} trainable parameters")
    t0 = time.time()
    tl_hist[name] = fit(model, *make_loaders(train_tf, eval_tf), opt, EPOCHS_TL, DEVICE,
                        scheduler=sched, amp=HW["amp"])
    tl_time[name] = time.time() - t0
    tl_models[name] = model

In [ ]:
plot_history(tl_hist, title="Figure 6.3 - Transfer learning strategies (dashed = train, solid = validation)")
names = list(strategies)
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
colors = ["#9aa0a6", "#fbbc04", "#188038"]
for ax, vals, title, fmt in [(axes[0], [tl_hist[n]["val_acc"][-1] for n in names], "validation accuracy", "{:.3f}"),
                             (axes[1], [tl_time[n] for n in names], f"training time on {DEVICE} (s)", "{:.0f}s"),
                             (axes[2], [tl_trainable[n] for n in names], "trainable parameters", "{:,}")]:
    bars = ax.bar(names, vals, color=colors)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), fmt.format(v), ha="center", va="bottom", fontsize=9)
    ax.set_title(title); ax.grid(axis="y", alpha=0.3)
plt.suptitle("Figure 6.4 - Pretrained features win with little data", fontweight="bold")
plt.tight_layout(); plt.show()

### 6.1 How much labelled data does pretraining save?
A linear probe is cheap: we extract the 512-number ResNet features **once**, then train only a linear classifier. That lets us
measure accuracy as a function of the number of labelled images per class, averaged over 3 random draws.

In [ ]:
extractor = make_resnet("linear probe")
extractor.fc = nn.Identity()
extractor.eval()


@torch.no_grad()
def extract(images, tf=eval_tf, model=extractor):
    feats = []
    for i in range(0, len(images), 64):
        x = torch.stack([tf(im) for im in images[i:i + 64]]).to(DEVICE)
        feats.append(model(x).float().cpu())
    return torch.cat(feats)


F_train, F_val = extract(train_imgs), extract(val_imgs)
y_train_t, y_val_t = torch.tensor(train_labels), torch.tensor(val_labels)


def probe_accuracy(Ftr, ytr, Fva, yva, steps=200):
    clf = nn.Linear(Ftr.shape[1], len(CLASSES))
    opt = torch.optim.Adam(clf.parameters(), 1e-2, weight_decay=1e-4)
    for _ in range(steps):
        loss = F.cross_entropy(clf(Ftr), ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    return (clf(Fva).argmax(1) == yva).float().mean().item()


shots = [k for k in (1, 2, 5, 10, 20, 50, 100, 200) if k <= PER_CLASS_TRAIN]
curve = []
for k in shots:
    accs = []
    for seed in range(3):
        g = torch.Generator().manual_seed(seed)
        idx = torch.cat([torch.where(y_train_t == c)[0][torch.randperm(PER_CLASS_TRAIN, generator=g)[:k]] for c in range(10)])
        accs.append(probe_accuracy(F_train[idx], y_train_t[idx], F_val, y_val_t))
    curve.append((np.mean(accs), np.std(accs)))
m_, s_ = np.array(curve).T
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.errorbar(shots, m_, yerr=s_, marker="o", lw=2, capsize=4, color="#188038", label="pretrained ResNet-18 + linear probe")
ax.axhline(tl_hist["scratch"]["val_acc"][-1], color="#9aa0a6", ls="--",
           label=f"ResNet-18 from scratch, all {PER_CLASS_TRAIN} images/class")
ax.axhline(0.1, color="#d93025", ls=":", label="chance (10 classes)")
ax.set_xscale("log"); ax.set_xticks(shots, shots); ax.set_xlabel("labelled images per class")
ax.set_ylabel("validation accuracy"); ax.set_ylim(0, 1.02); ax.grid(alpha=0.3); ax.legend(loc="lower right")
ax.set_title("Figure 6.5 - Data efficiency of transfer learning", fontweight="bold")
plt.show()

> **Be critical.** Imagenette's classes are *part of* ImageNet, so this is the best case for transfer. On industrial images
> (metal surfaces, PCBs, X-rays) the gain is smaller but usually still large, because low-level features transfer well.
> The challenge in Section 12 asks you to measure this on your own objects.

### 6.2 Final test-set evaluation (done once)
We choose the best strategy **on validation accuracy**, and only then touch the test set.

In [ ]:
best = max(tl_hist, key=lambda n: tl_hist[n]["val_acc"][-1])
test_loader = DataLoader(ImageList(test_imgs, test_labels, eval_tf), batch_size=BATCH * 2)
logits, y_true = predict(tl_models[best], test_loader, DEVICE)
y_pred = logits.argmax(1)
print(f"Selected on validation: {best!r} -> TEST accuracy {(y_pred == y_true).mean():.3f}")
plot_confusion(confusion_matrix(y_true, y_pred, 10), CLASSES, title=f"Figure 6.6 - Test confusion matrix ({best})")

## 7. Grad-CAM: what is the network looking at?

An inspection system must be trustworthy, and "98% accurate" is not enough if the model looks at the wrong thing
(for example a watermark, or the conveyor belt instead of the part). **Grad-CAM** (Selvaraju et al., 2017) produces a
heat map of the image regions that raised the score of a class $c$:

1. Take the feature maps $A^k$ of the last convolutional layer (for ResNet-18: 512 maps of 7×7 at 224 px input).
2. Compute how much each map matters for class $c$: the spatially averaged gradient
   $\displaystyle \alpha_k^c = \frac{1}{Z}\sum_{i,j}\frac{\partial y^c}{\partial A^k_{ij}}$.
3. Combine and keep only positive evidence: $L^c = \text{ReLU}\left(\sum_k \alpha_k^c A^k\right)$, then upsample to the image size.

In [ ]:
class GradCAM:
    """Grad-CAM for any CNN layer, using forward and backward hooks."""

    def __init__(self, model, layer):
        self.model = model
        self.acts = self.grads = None
        layer.register_forward_hook(self._save)

    def _save(self, _module, _inp, out):
        self.acts = out
        if out.requires_grad:  # skip during no-grad inference (e.g. predict())
            out.register_hook(lambda g: setattr(self, "grads", g))

    def __call__(self, x, class_idx=None):
        self.model.eval()
        x = x.clone().requires_grad_(True)  # ensures gradients flow even through a frozen backbone
        logits = self.model(x)
        probs = logits.softmax(1)[0]
        c = int(probs.argmax()) if class_idx is None else class_idx
        self.model.zero_grad()
        logits[0, c].backward()
        alpha = self.grads.mean(dim=(2, 3), keepdim=True)                      # step 2
        cam = F.relu((alpha * self.acts).sum(1, keepdim=True))                  # step 3
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), c, probs.detach().cpu().numpy()


def overlay(rgb01, cam, alpha=0.45):
    heat = plt.cm.jet(cam)[..., :3]
    return np.clip((1 - alpha) * rgb01 + alpha * heat, 0, 1)


imnet_tf = T.Compose([T.Resize(224, antialias=True), T.CenterCrop(224), *to_float])
cam_imnet = GradCAM(imnet_model, imnet_model.layer4)
demo_images = {"cat (chelsea)": skimage.data.chelsea(), "coffee": skimage.data.coffee(),
               "astronaut": skimage.data.astronaut(), "rocket": skimage.data.rocket()}
fig, axes = plt.subplots(2, 4, figsize=(17, 8.5))
for j, (name, arr) in enumerate(demo_images.items()):
    x = imnet_tf(Image.fromarray(arr))[None].to(DEVICE)
    cam, c, probs = cam_imnet(x)
    rgb = denormalize(x[0])
    axes[0, j].imshow(rgb); axes[0, j].set_title(name)
    axes[1, j].imshow(overlay(rgb, cam)); axes[1, j].set_title(f"{IMNET_CLASSES[c]} ({probs[c]:.2f})")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Figure 7.1 - Grad-CAM for the top ImageNet class" + ("" if PRETRAINED else "  [random weights: offline]"),
             fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: explain any of the top-5 classes
The same image can contain evidence for several classes. Grad-CAM shows *where* each one comes from.

In [ ]:
def cam_demo(image="astronaut", rank=1, alpha=0.45):
    x = imnet_tf(Image.fromarray(demo_images[image]))[None].to(DEVICE)
    _, _, probs = cam_imnet(x)
    top5 = np.argsort(probs)[::-1][:5]
    cam, c, _ = cam_imnet(x, int(top5[rank - 1]))
    rgb = denormalize(x[0])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1, 1, 1.2]})
    axes[0].imshow(rgb); axes[0].axis("off"); axes[0].set_title("input")
    axes[1].imshow(overlay(rgb, cam, alpha)); axes[1].axis("off"); axes[1].set_title(f"Grad-CAM: {IMNET_CLASSES[c]}")
    axes[2].barh([IMNET_CLASSES[i] for i in top5][::-1], probs[top5][::-1],
                 color=["#188038" if i == c else "#9aa0a6" for i in top5][::-1])
    axes[2].set_title("top-5 probabilities"); axes[2].set_xlim(0, 1)
    plt.tight_layout(); plt.show()


interact(cam_demo, image=list(demo_images), rank=widgets.IntSlider(1, 1, 5),
         alpha=widgets.FloatSlider(0.45, min=0.0, max=1.0, step=0.05))

In [ ]:
ft_model = tl_models[best]
cam_ft = GradCAM(ft_model, ft_model.layer4)
correct = np.where(y_pred == y_true)[0][:4]
wrong = np.where(y_pred != y_true)[0][:4]
fig, axes = plt.subplots(2, 4, figsize=(16, 8.4))
for row, idxs, color in [(0, correct, "#188038"), (1, wrong, "#d93025")]:
    for j in range(4):
        ax = axes[row, j]
        ax.axis("off")
        if j >= len(idxs):
            continue
        i = idxs[j]
        x = eval_tf(test_imgs[i])[None].to(DEVICE)
        cam, c, probs = cam_ft(x)
        ax.imshow(overlay(denormalize(x[0]), cam))
        ax.set_title(f"true: {CLASSES[y_true[i]]}\npred: {CLASSES[c]} ({probs[c]:.2f})", color=color, fontsize=10)
plt.suptitle(f"Figure 7.2 - Grad-CAM on our {best} model: correct (top) and wrong (bottom) test predictions",
             fontweight="bold")
plt.tight_layout(); plt.subplots_adjust(hspace=0.3); plt.show()

## 8. Beyond CNNs: Vision Transformers in one page

A **Vision Transformer** (ViT; Dosovitskiy et al., 2021) cuts the image into 16×16 patches, turns each patch into a vector
("token"), adds a position code, and lets every token exchange information with every other token through **self-attention**.

| | CNN | Vision Transformer |
|---|---|---|
| Built-in assumptions | locality, weight sharing | almost none; learns them from data |
| Receptive field | grows slowly with depth | global from the first layer |
| Data needed from scratch | moderate | very large (or self-supervised pretraining) |
| Where you meet it in this course | detectors, U-Net (Weeks 8–11) | SAM, DINOv3, CLIP and VLMs (Weeks 11–13) |

In [ ]:
cat = np.array(Image.fromarray(skimage.data.chelsea()).resize((224, 224)))
P = 16
fig = plt.figure(figsize=(17, 5))
ax0 = fig.add_axes([0.0, 0.05, 0.3, 0.85])
ax0.imshow(cat)
for k in range(0, 225, P):
    ax0.axhline(k - 0.5, color="w", lw=0.8); ax0.axvline(k - 0.5, color="w", lw=0.8)
ax0.set_title("224x224 image -> 14x14 = 196 patches of 16x16"); ax0.axis("off")
for t in range(14):
    r, c = divmod(t + 84, 14)
    ax = fig.add_axes([0.33 + t * 0.047, 0.35, 0.042, 0.3])
    ax.imshow(cat[r * P:(r + 1) * P, c * P:(c + 1) * P]); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"#{t + 84}", fontsize=8)
fig.text(0.66, 0.2, "each patch -> linear projection -> token (+ position)  ->  transformer layers (self-attention)",
         ha="center", fontsize=11)
fig.suptitle("Figure 8.1 - A ViT reads an image as a sequence of patch tokens", fontweight="bold")
plt.show()

**Figure 8.2 — The model zoo, today.** Accuracy vs compute for every ImageNet classifier shipped with your installed torchvision, read from the model metadata (no download needed). CNNs and transformers now reach similar accuracy; the right choice depends on your latency, memory and data budget.

In [ ]:
from torchvision.models import get_model_weights, list_models

rows = []
for name in list_models(module=torchvision.models):
    try:
        meta = get_model_weights(name).DEFAULT.meta
        rows.append((name, meta["_metrics"]["ImageNet-1K"]["acc@1"], meta["_ops"], meta["num_params"] / 1e6))
    except Exception:  # noqa: BLE001 - skip models without standard metadata
        pass
families = {"resnet": "ResNet", "resnext": "ResNet", "wide": "ResNet", "efficientnet": "EfficientNet",
            "mobilenet": "Mobile (MobileNet, ShuffleNet, MnasNet)", "shufflenet": "Mobile (MobileNet, ShuffleNet, MnasNet)",
            "mnasnet": "Mobile (MobileNet, ShuffleNet, MnasNet)", "squeezenet": "Mobile (MobileNet, ShuffleNet, MnasNet)",
            "convnext": "ConvNeXt", "vit": "ViT / Swin / MaxViT", "swin": "ViT / Swin / MaxViT", "maxvit": "ViT / Swin / MaxViT",
            "regnet": "RegNet", "vgg": "Classic (AlexNet, VGG, Inception, DenseNet)",
            "alexnet": "Classic (AlexNet, VGG, Inception, DenseNet)", "densenet": "Classic (AlexNet, VGG, Inception, DenseNet)",
            "inception": "Classic (AlexNet, VGG, Inception, DenseNet)", "googlenet": "Classic (AlexNet, VGG, Inception, DenseNet)"}
fig, ax = plt.subplots(figsize=(12, 6))
fam_colors = {}
for name, acc, gflops, mparams in rows:
    fam = next((v for k, v in families.items() if name.startswith(k)), "other")
    color = fam_colors.setdefault(fam, plt.cm.tab10(len(fam_colors) % 10))
    ax.scatter(gflops, acc, s=12 + 3 * mparams, color=color, alpha=0.75, edgecolor="k", lw=0.4)
    if name in {"resnet18", "mobilenet_v3_small", "efficientnet_b0", "convnext_tiny", "vit_b_16", "alexnet", "vgg16"}:
        ax.annotate(name, (gflops, acc), xytext=(4, 4), textcoords="offset points", fontsize=9)
for fam, color in fam_colors.items():
    ax.scatter([], [], color=color, label=fam)
ax.set_xscale("log"); ax.set_xlabel("GFLOPs per image (log)"); ax.set_ylabel("ImageNet top-1 accuracy (%)")
ax.grid(alpha=0.3); ax.legend(title="family (marker size = parameters)", fontsize=8)
ax.set_title(f"Figure 8.2 - {len(rows)} torchvision classifiers: accuracy vs compute", fontweight="bold")
plt.show()

## 9. Classical vs learned, revisited

In Weeks 4 and 5 we met two limits of hand-crafted methods. Let's revisit them with learned features.

### 9.1 Finding *similar* images (retrieval)
In Week 5, SIFT and HOG matched **the same object** well, but they cannot tell that two *different* garbage trucks belong to the same class.
Below, each query image retrieves its 5 nearest neighbours in the validation set using HOG (on greyscale) and using pretrained ResNet features.

In [ ]:
def hog_desc(img):
    g = np.array(img.convert("L").resize((64, 64)))
    return hog(g, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2))


H_val = torch.tensor(np.stack([hog_desc(im) for im in val_imgs]), dtype=torch.float32)
reps = {"HOG": F.normalize(H_val, dim=1), "ResNet-18 features": F.normalize(F_val, dim=1)}


def precision_at_5(R):
    sim = R @ R.T
    sim.fill_diagonal_(-1)
    nn5 = sim.topk(5, dim=1).indices
    return (y_val_t[nn5] == y_val_t[:, None]).float().mean().item()


queries = [int(np.where(val_labels == c)[0][0]) for c in (6, 3, 7)]
fig, axes = plt.subplots(6, 6, figsize=(13, 13.5))
for q, qi in enumerate(queries):
    for r, (name, R) in enumerate(reps.items()):
        row = axes[2 * q + r]
        sim = R @ R[qi]
        sim[qi] = -1
        row[0].imshow(val_imgs[qi]); row[0].set_title("query" if r == 0 else "", fontweight="bold")
        row[0].set_ylabel(name, fontsize=9)
        for j, i in enumerate(sim.topk(5).indices.tolist()):
            row[j + 1].imshow(val_imgs[i])
            row[j + 1].set_title(CLASSES[val_labels[i]], fontsize=8,
                                 color="#188038" if val_labels[i] == val_labels[qi] else "#d93025")
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Figure 9.1 - Top-5 retrieval: HOG (odd rows) vs learned features (even rows); green = same class",
             fontweight="bold")
plt.tight_layout(); plt.show()

p5 = {name: precision_at_5(R) for name, R in reps.items()}
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(list(p5), list(p5.values()), color=["#fbbc04", "#188038"]); ax.set_xlim(0, 1)
for i, v in enumerate(p5.values()):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center")
ax.set_title("Figure 9.2 - Precision@5 over all validation queries", fontweight="bold")
plt.show()

### 9.2 Segmenting a *person* (not just bright pixels)
In Week 4, Otsu thresholding separated bright from dark. It has no notion of *what* a pixel belongs to. A pretrained
semantic-segmentation network (DeepLabV3 with a MobileNetV3 backbone, 42 MB, runs on a laptop CPU) labels every pixel with a class.
You will study how these networks work in Week 11.

In [ ]:
seg_w = torchvision.models.segmentation.DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT
seg_model, SEG_OK = load_pretrained(torchvision.models.segmentation.deeplabv3_mobilenet_v3_large, seg_w)
seg_model = seg_model.to(DEVICE).eval()
astro = skimage.data.astronaut()
with torch.no_grad():
    out = seg_model(seg_w.transforms()(torch.from_numpy(astro).permute(2, 0, 1))[None].to(DEVICE))["out"][0]
labels_map = F.interpolate(out[None], size=astro.shape[:2], mode="bilinear")[0].argmax(0).cpu().numpy()
person = labels_map == seg_w.meta["categories"].index("person")
gray = cv2.cvtColor(astro, cv2.COLOR_RGB2GRAY)
otsu = gray > threshold_otsu(gray)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
axes[0].imshow(astro); axes[0].set_title("input")
axes[1].imshow(otsu, cmap="gray"); axes[1].set_title("Week 4: Otsu threshold (bright vs dark)")
axes[2].imshow(labels_map, cmap="tab20", vmin=0, vmax=20); axes[2].set_title("DeepLabV3: class of every pixel")
axes[3].imshow(astro); axes[3].imshow(np.ma.masked_where(~person, person), cmap="autumn", alpha=0.55)
axes[3].set_title("learned: 'person' mask")
for ax in axes:
    ax.axis("off")
plt.suptitle("Figure 9.3 - Hand-crafted vs learned segmentation" + ("" if SEG_OK else "  [random weights: offline]"),
             fontweight="bold")
plt.tight_layout(); plt.show()

Matching across *different instances* of a class, and matching under large viewpoint changes, has also moved to learned
features: **SuperPoint + LightGlue** for keypoints and **DINOv2/DINOv3** for dense features. See the further reading.

## 10. Will it fit on my 4 GB GPU?

GPU memory during training holds the weights, the gradients, the optimiser state and, above all, the **activations**
of every layer for the whole batch. Memory therefore grows roughly linearly with **batch size** and with **image area**.
Mixed precision (`amp`, 16-bit activations) roughly halves the activation memory. The cell below measures ResNet-18 on *your* machine.

Rules of thumb for a 4 GB laptop GPU: ResNet-18/34 or EfficientNet-B0 at 224 px with batch 32–64 and AMP is comfortable.
If you hit `CUDA out of memory`: halve the batch size, reduce the image size, enable AMP, or freeze the backbone.

In [ ]:
def benchmark(batch_sizes, img=IMG, amp=HW["amp"], steps=3):
    model = torchvision.models.resnet18(num_classes=10).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), 1e-3)
    scaler = torch.amp.GradScaler("cuda") if amp and HW["cuda"] else None
    rows = []
    for b in batch_sizes:
        x = torch.randn(b, 3, img, img, device=DEVICE)
        y = torch.randint(0, 10, (b,), device=DEVICE)
        if HW["cuda"]:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        for step in range(steps + 1):
            if step == 1:
                t0 = time.time()
            with torch.autocast(device_type=DEVICE.type, enabled=scaler is not None):
                loss = F.cross_entropy(model(x), y)
            opt.zero_grad()
            if scaler:
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
        if HW["cuda"]:
            torch.cuda.synchronize()
        dt = (time.time() - t0) / steps
        mem = torch.cuda.max_memory_allocated() / 1024 ** 3 if HW["cuda"] else float("nan")
        rows.append((b, b / dt, mem))
        print(f"batch {b:4d}: {b / dt:7.1f} images/s" + (f", peak GPU memory {mem:.2f} GB" if HW["cuda"] else ""))
    return rows


sizes = [2, 4] if SMOKE else ([16, 32, 64, 128] if HW["cuda"] else [8, 16, 32])
bench = benchmark(sizes)
fig, axes = plt.subplots(1, 2 if HW["cuda"] else 1, figsize=(13 if HW["cuda"] else 7, 3.8), squeeze=False)
axes[0, 0].plot([r[0] for r in bench], [r[1] for r in bench], "o-", lw=2)
axes[0, 0].set_xlabel("batch size"); axes[0, 0].set_ylabel("training images / second"); axes[0, 0].grid(alpha=0.3)
axes[0, 0].set_title(f"Throughput on {HW['gpu_name'] or 'CPU'}")
if HW["cuda"]:
    axes[0, 1].plot([r[0] for r in bench], [r[2] for r in bench], "o-", lw=2, color="#d93025")
    axes[0, 1].axhline(4.0, color="k", ls="--", label="4 GB laptop GPU")
    axes[0, 1].set_xlabel("batch size"); axes[0, 1].set_ylabel("peak memory (GB)"); axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3); axes[0, 1].set_title(f"ResNet-18 training at {IMG} px, AMP={HW['amp']}")
plt.suptitle("Figure 10.1 - Measured on your machine", fontweight="bold")
plt.tight_layout(); plt.show()

## 11. Exercises (with self-checks)

Replace each `raise NotImplementedError`, then run the check below it. The checks test *behaviour*, not a specific solution.

### E1. Your own augmentation pipeline
Return a `torchvision.transforms.v2` pipeline that takes a PIL image and outputs a normalised float tensor of shape `(3, size, size)`.
It must include **at least one random geometric** transform and **one random photometric** transform that suit an *industrial inspection*
setting of your choice. Add a comment justifying each choice.

In [ ]:
def build_train_transform(size):
    # TODO: return T.Compose([...])
    raise NotImplementedError

In [ ]:
def _test_e1():
    tf = build_train_transform(96)
    a, b = tf(train_imgs[0]), tf(train_imgs[0])
    assert torch.is_tensor(a) and a.dtype == torch.float32, "the pipeline must end with a float32 tensor (T.ToImage + T.ToDtype)"
    assert tuple(a.shape) == (3, 96, 96), f"expected shape (3, 96, 96), got {tuple(a.shape)}"
    assert not torch.allclose(a, b), "two calls on the same image gave identical results: add random transforms"
    mean = torch.stack([tf(im) for im in train_imgs[:20]]).mean().item()
    assert abs(mean) < 0.75, f"average value {mean:.2f} is far from 0: did you normalise with the ImageNet mean/std?"


check("E1 build_train_transform", _test_e1)

### E2. Discriminative learning rates
When fine-tuning, the new head (random weights) should learn faster than the pretrained backbone. Return a list of **two
parameter groups** for `torch.optim.AdamW`: the `fc` parameters with `head_lr`, and every other parameter with `backbone_lr`.

In [ ]:
def make_param_groups(model, backbone_lr=1e-4, head_lr=1e-3):
    # TODO: return [{"params": [...], "lr": ...}, {"params": [...], "lr": ...}]
    raise NotImplementedError

In [ ]:
def _test_e2():
    model = torchvision.models.resnet18(num_classes=10)
    groups = make_param_groups(model, 1e-4, 1e-3)
    assert isinstance(groups, list) and len(groups) == 2, "return a list with exactly two groups"
    seen = [id(p) for g in groups for p in g["params"]]
    assert len(seen) == len(set(seen)), "a parameter appears in both groups"
    assert set(seen) == {id(p) for p in model.parameters()}, "every model parameter must be in exactly one group"
    fc_ids = {id(p) for p in model.fc.parameters()}
    for g in groups:
        ids = {id(p) for p in g["params"]}
        expected = 1e-3 if ids == fc_ids else 1e-4
        assert g["lr"] == expected, f"group with {len(ids)} tensors has lr {g['lr']}, expected {expected}"
    assert any({id(p) for p in g["params"]} == fc_ids for g in groups), "one group must hold exactly the fc parameters"
    torch.optim.AdamW(groups)  # must be accepted by the optimiser


check("E2 make_param_groups", _test_e2)

### E3. Early stopping
Training stops when the validation loss has **not improved for `patience` consecutive epochs** (or when the list ends).
Return the **1-based epoch** whose weights you would keep: the best epoch seen *before* stopping.

In [ ]:
def early_stop_epoch(val_losses, patience=3):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    cases = [([1.0, 0.8, 0.7, 0.75, 0.72, 0.71, 0.6], 3, 3),
             ([1.0, 0.9, 0.8, 0.7], 2, 4),
             ([0.5, 0.6, 0.4, 0.45, 0.46], 2, 3),
             ([0.5, 0.6, 0.7, 0.3], 2, 1)]
    for losses, patience, expected in cases:
        got = early_stop_epoch(losses, patience)
        assert got == expected, f"losses={losses}, patience={patience}: expected epoch {expected}, got {got}"


check("E3 early_stop_epoch", _test_e3)

### E4. Experiment (no self-check)
Train a **partial fine-tune**: freeze everything except `layer4` and `fc` of ResNet-18. Compare its validation accuracy,
training time and trainable-parameter count with the three strategies in Figure 6.4. When would you choose it?

In [ ]:
# E4: your experiment here

## 12. Challenge: a classifier for *your* objects

Build a 3–5 class classifier for objects on your desk (e.g. screws, nuts, washers, resistors, or empty vs full bottle caps),
using your **laptop webcam** and transfer learning.

1. Capture 30–50 images per class with the helper below (set `RUN_WEBCAM = True`, locally only). Vary position, rotation and lighting.
   In Colab, upload photos into `data/my_objects/<class_name>/` instead.
2. Fine-tune ResNet-18 using this notebook's code (`torchvision.datasets.ImageFolder` loads the folder structure directly).
3. Report validation accuracy, a confusion matrix and **Grad-CAM** for 4 images. Is the model looking at the object or at the background?
4. **Domain shift test:** change the background or lighting and re-test. What happens, and how would you fix it (Weeks 2, 13)?

In [ ]:
RUN_WEBCAM = False   # set True on your laptop


def capture_dataset(class_names, n_per_class=40, out_dir=None, camera=0, delay_s=0.25):
    """Capture images from a webcam into out_dir/<class>/ (press Enter to start each class)."""
    out_dir = Path(out_dir or data_dir("my_objects"))
    cap = cv2.VideoCapture(camera)
    if not cap.isOpened():
        print("No webcam found.")
        return
    try:
        for name in class_names:
            (out_dir / name).mkdir(parents=True, exist_ok=True)
            input(f"Place '{name}' in view and press Enter...")
            for k in range(n_per_class):
                ok, frame = cap.read()
                if ok:
                    cv2.imwrite(str(out_dir / name / f"{name}_{k:03d}.jpg"), frame)
                time.sleep(delay_s)
            print(f"  saved {n_per_class} images of {name}")
    finally:
        cap.release()


if RUN_WEBCAM:
    capture_dataset(["screw", "nut", "washer"])

## 13. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'Training loss keeps falling but validation loss starts rising. What is happening?',
        "options": [
            'Underfitting',
            'Overfitting',
            'The learning rate is zero',
            'The test set is too small',
        ],
        "answer": '9daf354fd4',
        "explain": 'The model memorises the training set. Use augmentation, regularisation, more data or early stopping.',
    },
    {
        "q": "What is a 'linear probe'?",
        "options": [
            'Training every layer from random weights',
            'Freezing the pretrained backbone and training only a new final layer',
            'Measuring the learning rate',
            'A type of data augmentation',
        ],
        "answer": 'f82e18643e',
        "explain": 'It is fast, needs little data, and even works on a CPU.',
    },
    {
        "q": 'Why must you normalise inputs with the ImageNet mean/std when using an ImageNet-pretrained model?',
        "options": [
            'It makes images look nicer',
            'The pretrained weights expect inputs with the statistics they were trained on',
            'It is required by PyTorch',
            'It reduces the file size',
        ],
        "answer": '6b6acf91b1',
        "explain": 'Different input statistics shift every activation and silently degrade accuracy.',
    },
    {
        "q": 'When should the test set be used?',
        "options": [
            'After every epoch to pick the best model',
            'To tune the learning rate',
            'Once, after all design decisions are made',
            'Never',
        ],
        "answer": '77564ea29c',
        "explain": 'Anything tuned on the test set makes the reported accuracy optimistic.',
    },
    {
        "q": 'In Grad-CAM, what are the channel weights alpha_k?',
        "options": [
            'The ImageNet class probabilities',
            'The spatially averaged gradients of the class score with respect to each feature map',
            'Random numbers',
            'The first-layer filter values',
        ],
        "answer": 'f68e02862a',
        "explain": 'They measure how much each feature map raises the score of the chosen class.',
    },
    {
        "q": 'You are classifying analogue gauge readings. Which augmentation is WRONG?',
        "options": [
            'Small brightness changes',
            'Slight blur',
            'Large random rotations',
            'Small random crops',
        ],
        "answer": '94e9217c20',
        "explain": 'Rotating a gauge changes its reading, so the label is no longer correct.',
    },
    {
        "q": "Training on your 4 GB GPU fails with 'CUDA out of memory'. What is the best first fix?",
        "options": [
            'Increase the batch size',
            'Halve the batch size and/or enable mixed precision (AMP)',
            'Increase the image size',
            'Switch to a bigger model',
        ],
        "answer": '06ac9e6f46',
        "explain": 'Activation memory scales with batch size and image area; AMP roughly halves it.',
    },
    {
        "q": 'Why does transfer learning need far fewer labelled images?',
        "options": [
            'Pretrained networks do not need labels',
            'Early and middle layers already encode general features (edges, textures, parts)',
            'It uses a larger learning rate',
            'It skips the validation set',
        ],
        "answer": '4dda5f449d',
        "explain": 'Only the task-specific part has to be learned from your small dataset.',
    },
], title='Week 06B quiz')

## 14. Exit ticket (reflection)
1. Your project has 80 labelled images of a defect. Which transfer strategy would you start with, and which three augmentations? Why?
2. Grad-CAM shows your "defect" classifier focusing on a label sticker that appears only on defective parts. What went wrong, and how do you fix the **data**?
3. After Week 6, name one task where you would still use a classical pipeline from Weeks 3–5, and one where you would not.

## 15. What's new (2025–26)

> - **Foundation backbones as frozen feature extractors.** Self-supervised ViTs such as **DINOv2** (2023) and **DINOv3** (2025) give
>   features that often beat supervised ImageNet backbones in a linear probe, especially for dense tasks like segmentation and matching.
> - **Parameter-efficient fine-tuning** (e.g. LoRA adapters) trains under 1% of a large model's weights, which makes big backbones practical on small GPUs.
> - **Model hubs.** `timm` and the Hugging Face Hub host thousands of pretrained vision models. Always check each model's licence before commercial use.
> - **Learned matching.** SuperPoint + LightGlue and dense matchers have largely replaced SIFT + RANSAC in hard, wide-baseline cases (Week 5 revisited).

## 16. References and further exploration

**Seminal papers**
- He, Zhang, Ren & Sun (2016). *Deep Residual Learning for Image Recognition* (ResNet). CVPR. [arXiv:1512.03385](https://arxiv.org/abs/1512.03385)
- Yosinski, Clune, Bengio & Lipson (2014). *How transferable are features in deep neural networks?* NeurIPS. [arXiv:1411.1792](https://arxiv.org/abs/1411.1792)
- Srivastava et al. (2014). *Dropout: A Simple Way to Prevent Neural Networks from Overfitting.* JMLR 15. [jmlr.org](https://jmlr.org/papers/v15/srivastava14a.html)
- Ioffe & Szegedy (2015). *Batch Normalization.* ICML. [arXiv:1502.03167](https://arxiv.org/abs/1502.03167)
- Loshchilov & Hutter (2019). *Decoupled Weight Decay Regularization* (AdamW). ICLR. [arXiv:1711.05101](https://arxiv.org/abs/1711.05101)
- Smith (2018). *A disciplined approach to neural network hyper-parameters* (one-cycle). [arXiv:1803.09820](https://arxiv.org/abs/1803.09820)
- Selvaraju et al. (2017). *Grad-CAM: Visual Explanations from Deep Networks.* ICCV. [arXiv:1610.02391](https://arxiv.org/abs/1610.02391)
- Dosovitskiy et al. (2021). *An Image is Worth 16x16 Words* (ViT). ICLR. [arXiv:2010.11929](https://arxiv.org/abs/2010.11929)
- Shorten & Khoshgoftaar (2019). *A survey on Image Data Augmentation for Deep Learning.* J. Big Data 6. [doi:10.1186/s40537-019-0197-0](https://doi.org/10.1186/s40537-019-0197-0)
- Howard et al. (2019). *Searching for MobileNetV3.* [arXiv:1905.02244](https://arxiv.org/abs/1905.02244); Chen et al. (2017). *Rethinking Atrous Convolution* (DeepLabV3). [arXiv:1706.05587](https://arxiv.org/abs/1706.05587)
- Oquab et al. (2023). *DINOv2.* [arXiv:2304.07193](https://arxiv.org/abs/2304.07193); DINOv3 (Meta AI, 2025): [github.com/facebookresearch/dinov3](https://github.com/facebookresearch/dinov3)
- Lindenberger, Sarlin & Pollefeys (2023). *LightGlue.* ICCV. [arXiv:2306.13643](https://arxiv.org/abs/2306.13643)

**Textbooks and guides**
- Prince (2023). *Understanding Deep Learning*, ch. 8 (measuring performance), 9 (regularisation), 10–11 (CNNs, ResNets), 12 (transformers). [udlbook.github.io](https://udlbook.github.io/udlbook/)
- Szeliski (2022). *Computer Vision: Algorithms and Applications*, 2nd ed., §5.3–5.5. [szeliski.org/Book](https://szeliski.org/Book/)
- Karpathy (2019). *A Recipe for Training Neural Networks.* [karpathy.github.io](http://karpathy.github.io/2019/04/25/recipe/)
- CS231n notes: *Transfer Learning* and *Neural Networks Part 3*. [cs231n.github.io](https://cs231n.github.io/transfer-learning/)
- PyTorch tutorial: *Transfer Learning for Computer Vision*. [docs.pytorch.org](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- torchvision model zoo (accuracy, size, licences). [docs.pytorch.org/vision](https://docs.pytorch.org/vision/stable/models.html)
- `timm` (PyTorch Image Models). [huggingface.co/docs/timm](https://huggingface.co/docs/timm)

**Interactive**
- Distill: *Feature Visualization* (what neurons respond to). [distill.pub](https://distill.pub/2017/feature-visualization/)
- CNN Explainer. [poloclub.github.io/cnn-explainer](https://poloclub.github.io/cnn-explainer/)

**Data and images:** Imagenette © fast.ai (Apache-2.0), a subset of ImageNet (for non-commercial research/education).
`skimage.data` images: chelsea, coffee, astronaut (NASA, public domain), rocket (scikit-image, public domain/CC0).

## 17. Key takeaways
- Keep **train / validation / test** separate. Tune on validation and report test once.
- Overfitting shows up as a growing train–validation gap. Fight it with **augmentation**, **dropout**, **weight decay**, **early stopping** and more data.
- Augmentations must preserve the label *for your application*.
- **Transfer learning** turns a few hundred images into a strong model. Start with a linear probe, then fine-tune with a small learning rate.
- **Grad-CAM** reveals *where* the model looks, which is essential for trustworthy inspection.
- On a 4 GB GPU, control memory with **batch size, image size and AMP**.

**Next week:** 7A integration lab, a complete inspection pipeline combining classical preprocessing (Weeks 1–5) with a CNN (Week 6), followed by the **midterm** (Weeks 1–6).